# Controlling the Output — Format, Length, Tone, and Structure

In [ ]:
# If you are running this on Google Colab, uncomment and run the line below first.
# !pip install -q boto3 openai tiktoken anthropic matplotlib

## How LLM calls work in this notebook

Every live API call goes through `LLMRouter` from `garage_helper`:

1. **Model resolution** — the router maps your model name to the right provider (`anthropic`, `openai`, `azure_openai`, `bedrock_claude`, `gemini`, etc.) automatically.
2. **Stop sequences** — supported natively by Claude and OpenAI APIs. Pass them as `stop_sequences=[...]` in `router.generate()` kwargs and they are forwarded directly to the active provider.
3. **Temperature** — pass `temperature=0.0` (or any float) as a kwarg; the router forwards it to the provider request unchanged.
4. **`generate_response()`** — returns an `LLMResponse` with `.text`, `.input_tokens`, `.output_tokens`, and `.stop_reason` so you can inspect exactly why generation stopped.

These notebooks have been tested with **Claude** (via Anthropic direct API and AWS Bedrock) and **GPT** models (via Azure OpenAI and direct OpenAI). The `stop_reason` field is normalised across providers — `"end_turn"`, `"max_tokens"`, or `"stop_sequence"`.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath("../.."))

from garage_helper import setup_llm, LLMRouter

# Contributors: add DEFAULT_LLM_MODEL (and any provider credentials) to a .env at the repo root.
# Learners: setup_llm() will run an interactive wizard to pick a provider and enter credentials.
MODEL  = setup_llm()
router = LLMRouter(default_model=MODEL, verbose=False)

def ask(system: str, user: str, max_tokens: int = 300, **kwargs) -> str:
    return router.generate(user, model=MODEL, system=system, max_tokens=max_tokens, **kwargs)

## The model is not a vending machine — output shape is not automatic

In the previous notebooks we covered how to specify *what* you want: which concept, which task, which persona. The previous notebooks — Zero-Shot and Few-Shot — focused on getting the right answer. This one is about getting the right *shape*.

Here is a frustration every developer runs into early. The model gives you a genuinely correct, well-reasoned answer — but it comes back as four paragraphs of flowing prose when you needed a JSON object. Or three bullet points when you needed a table. Or 600 words when the UI has room for 80. The content is right; the package is wrong.

This is not a model failure. The model has no idea what your UI looks like, what downstream code will parse the response, or what your reader's attention span is. That is your job to communicate.

Output control is the skill of giving the model a precise specification of the container before it fills it. Done well, you stop post-processing model output and start using it directly. Done badly, you end up with a fragile regex trying to extract structured data from prose — at 3am, in production.

## Concept 1 — Format control: prose, markdown, tables, code blocks

The model can produce any text format fluently — plain prose, markdown with headers and bullets, HTML, code blocks, CSV, numbered lists, comparison tables. It just needs to know which one you want.

**The analogy:** a chef can plate the same dish as a casual bowl for lunch or a composed fine-dining plate for dinner. The ingredients are identical. The presentation is completely different and is entirely dictated by the occasion you described when you placed the order. The model is the same chef — it plated in whatever format it last saw most often in context unless you specify otherwise.

The clearest way to specify format is to name it explicitly and, for anything structural, describe the exact shape. "Respond in markdown" is weaker than "respond as a markdown table with columns: X, Y, Z." Name the format and describe the schema.

In [ ]:
topic  = "Compare Python lists and tuples."
system = "You are a concise technical writer. No preamble."

formats = [
    ("Prose",           topic),
    ("Bullet list",     topic + " Use a bullet list with at most 5 points."),
    ("Markdown table",  topic + " Respond as a markdown table with columns: Feature | List | Tuple."),
    ("Code + comment",  topic + " Respond as a Python code block only. Use inline comments to explain differences. No prose outside the code block."),
]

for label, prompt in formats:
    print(f"[Format: {label}]")
    print(ask(system, prompt))
    print("-" * 55)
    print()

print("Same question, four different containers. The format spec does all the shaping.")

## Concept 2 — JSON output: structured data you can actually parse

For applications — APIs, pipelines, databases, UIs — prose is almost always the wrong output format. You need data you can index, validate, and pass to the next function without string manipulation.

**The analogy:** think about ordering ingredients from a supplier. If they send you a letter describing what arrived in the truck, you have to read it and manually update your inventory. If they send an EDI file with a machine-readable item list, your system handles it automatically. JSON output from an LLM is the EDI file — it plugs directly into your system without a human in the loop.

Getting reliable JSON from a model requires three things together: tell the model the exact schema, tell it to output only JSON with no prose around it, and validate the output before trusting it. The third step matters — models occasionally add a stray sentence before the brace. Always wrap a `json.loads()` in a try/except.

In [ ]:
import json

system = """
You are a project task parser.
Output ONLY valid JSON — no markdown fences, no explanation, no text before or after the JSON.
Schema:
{
  "title":            string   (short task title, ≤60 chars),
  "category":         string   ("backend" | "frontend" | "infra" | "data" | "docs"),
  "priority":         string   ("HIGH" | "MEDIUM" | "LOW"),
  "tags":             [string] (1–4 lowercase tags),
  "estimated_hours":  number
}
""".strip()

raw_tasks = [
    "We need to migrate the old MySQL user table to Postgres before the end of sprint. Blocking the auth team. Probably 3 days of work.",
    "Update the README with new environment variable docs. Shouldn't take more than an hour.",
    "The dashboard chart is broken on mobile Safari. Designer flagged it as urgent. Needs CSS fix, maybe 2 hours.",
]

parsed_tasks = []
print(f"{'#':<3}  {'Parsing result':<20}  {'Title':<38}  {'Priority'}")
print("-" * 80)

for i, task_text in enumerate(raw_tasks, 1):
    raw = ask(system, task_text, max_tokens=200).strip()
    try:
        task = json.loads(raw)
        parsed_tasks.append(task)
        title = task.get("title", "")[:38]
        print(f"{i:<3}  {'✓ valid JSON':<20}  {title:<38}  {task.get('priority', '?')}")
    except json.JSONDecodeError as e:
        print(f"{i:<3}  {'✗ parse error':<20}  {str(e)[:60]}")

print()
print("Parsed tasks — ready to insert into a database or pass to an API:")
for task in parsed_tasks:
    print(f"  {task}")

## Concept 3 — Length control: word count, sentence count, token budgets

Left unconstrained, a language model will produce as much text as it takes to feel "complete" by its own standards — which is usually more than you need. This is not a flaw; it is the model optimising for thoroughness. Your job is to override that default with an explicit length target.

**The analogy:** a consultant asked for their opinion will give you a 45-minute presentation unless you say "give me the one-paragraph executive summary before the meeting starts." They have the long version and the short version. They default to the long version because that is what demonstrates value. You get the short version only when you ask for it.

Length instructions come in several flavours, each useful in different situations: word count ("under 80 words"), sentence count ("exactly three sentences"), structural count ("exactly four bullet points"), and API-level token limits (`max_tokens`). The first three go in the prompt — they describe the shape of the response. The last one is a hard ceiling enforced at the API level and does not substitute for prompt-level length guidance.

In [ ]:
base_question = "What is containerisation and why do developers use it?"
system = "You are a technical educator. No preamble."

length_specs = [
    ("None (default)",          base_question,                                                               400),
    ("≤30 words",               base_question + " Answer in 30 words or fewer.",                             80),
    ("Exactly 3 sentences",     base_question + " Answer in exactly three sentences.",                      150),
    ("4 bullets, 1 line each",  base_question + " Answer as exactly four bullet points. Each bullet is one short sentence.", 200),
]

print(f"{'Spec':<25}  {'Words':>6}  Response preview")
print("-" * 80)

for label, prompt, max_tok in length_specs:
    text    = ask(system, prompt, max_tokens=max_tok).strip()
    words   = len(text.split())
    preview = text[:70].replace("\n", " ") + ("..." if len(text) > 70 else "")
    print(f"{label:<25}  {words:>6}  {preview}")

print()
print("Length specs in the prompt shape the response. max_tokens is a ceiling, not a target.")
print("A prompt asking for 30 words with max_tokens=400 still returns ~30 words.")

## Concept 4 — Tone control: register, formality, and voice

Tone is often treated as vague or subjective, but it is actually highly controllable. The model has been trained on text spanning every register imaginable — academic papers, Slack messages, legal briefs, Reddit threads, children's books. It can match any of them given a clear enough description.

**The analogy:** a professional translator does not just convert words — they convert register. The same French sentence translated for a legal contract sounds completely different from the same sentence translated for a tourist brochure. Both translations are correct. The register you asked for determines everything about how the words are chosen. The model is that translator — the register is yours to specify.

Tone instructions work best when they are specific and combinatorial: name the **formality level** (casual, professional, academic), the **stance** (neutral, opinionated, encouraging, blunt), and optionally a **voice reference** ("as if written for a YC batch", "like a Stripe API doc", "like a senior engineer's code review comment"). The more anchors you give, the more recognisable the output register.

In [ ]:
content = "Explain why you should write tests before you write the code (TDD)."

tones = [
    (
        "Academic / formal",
        "Write in a formal academic register. Use third person. Cite the core reasoning without colloquialisms."
    ),
    (
        "Casual / conversational",
        "Write as if texting a developer friend. Short sentences, contractions fine, first person."
    ),
    (
        "Blunt / senior engineer",
        "Write like a senior engineer who has seen too many bugs from untested code. Opinionated, direct, zero hedging. One paragraph."
    ),
    (
        "Encouraging / mentor",
        "Write like a patient mentor talking to a junior developer who is skeptical about TDD. Warm, concrete, motivating. Two short paragraphs."
    ),
]

for label, tone_instruction in tones:
    system = f"{tone_instruction} No preamble."
    print(f"[Tone: {label}]")
    print(ask(system, content, max_tokens=180))
    print("-" * 60)
    print()

print("The factual content is identical. The register is entirely governed by the system prompt tone.")

## Concept 5 — Stop sequences: telling the model exactly where to stop

A stop sequence is a string you pass to the API that tells the model to halt generation the moment it produces that string. It is useful when you need to cut output at a precise boundary — after the closing brace of a JSON object, after the first code block, or at a sentinel token you define.

**The analogy:** stop sequences are like telling a transcriptionist "stop writing when you hear the word 'end of statement'." They do not care how long the answer was going to be — they stop the moment they hear the signal. You control the boundary, not the length.

Common uses: ensuring only one JSON object comes back (stop on `}\n`), extracting just the SQL query from a model that tends to add explanations (stop on `--`), or cleanly separating a structured response from an explanation section you will discard.

In [ ]:
system = (
    "You are a SQL assistant. "
    "First output the SQL query, then on a new line write '---' and then a one-sentence explanation."
)
user = "Write a query that returns the top 5 customers by total order value from an 'orders' table."

# Without stop sequence — we get both the SQL and the explanation
full_resp = router.generate_response(user, model=MODEL, system=system, max_tokens=250)

# With stop sequence — generation halts at '---', giving us only the SQL
sql_only_resp = router.generate_response(
    user, model=MODEL, system=system, max_tokens=250, stop_sequences=["---"]
)

print("WITHOUT stop sequence (full response):")
print("-" * 55)
print(full_resp.text)
print(f"Stop reason : {full_resp.stop_reason}")

print()
print("WITH stop_sequences=['---'] (SQL only):")
print("-" * 55)
print(sql_only_resp.text)
print(f"Stop reason : {sql_only_resp.stop_reason}")

print()
print("stop_reason='stop_sequence' confirms the model halted at our sentinel, not at end-of-turn.")
print("The SQL is now clean and directly executable — no post-processing needed.")

## Concept 6 — Temperature: the creativity dial

Temperature is an API parameter that controls how much randomness is injected into the model's token selection. At temperature 0 the model always picks the highest-probability token — deterministic, consistent, repetitive. At temperature 1 (the default for most models) it samples from the distribution — varied, sometimes surprising, occasionally wrong.

**The analogy:** imagine a jazz musician who has memorised a hundred standard songs perfectly. Temperature 0 is playing the song exactly as written, note for note, every performance identical. Temperature 1 is playing it with improvisation — the structure is the same but the fills change every night. Temperature above 1 is handing them a second whisky — interesting things happen, some of them brilliant, some of them not.

The practical rule of thumb: **low temperature for precision tasks** (JSON extraction, SQL generation, classification, factual Q&A), **higher temperature for creative tasks** (brainstorming, writing variation, ideation). Never go above 1 for production pipelines — the output becomes unreliable.

In [ ]:
system = "You are a creative copywriter. Be original."
user   = "Write a one-sentence tagline for a developer productivity app called Flowstate."

temperatures = [0.0, 0.3, 0.7, 1.0]

print(f"Task: '{user}'")
print(f"Running each temperature 3 times to show variance...")
print()

for temp in temperatures:
    print(f"[temperature = {temp}]")
    outputs = set()
    for _ in range(3):
        reply = router.generate(user, model=MODEL, system=system, max_tokens=60, temperature=temp)
        outputs.add(reply.strip())
    for line in sorted(outputs):
        marker = "=" if len(outputs) == 1 else "~"
        print(f"  {marker} {line}")
    unique_note = "(deterministic — all 3 runs identical)" if len(outputs) == 1 else f"({len(outputs)} distinct outputs across 3 runs)"
    print(f"  {unique_note}")
    print()

print("Low temperature: consistent and safe. High temperature: varied and creative.")
print("Use temperature=0 for anything that must be parsed or compared programmatically.")

## Putting it together — a complete output-controlled pipeline

All five controls wired into one pipeline: JSON schema + format instruction in the system prompt, length constraint in the user message, tone set by role, stop sequence as a safety net, and temperature at zero for a parsing task that must be deterministic.

In [ ]:
import json

system = """
You are a meeting notes parser.
Extract action items and output ONLY a JSON array — nothing before or after it.
Each element: {"owner": string, "action": string (≤80 chars), "deadline": string ("ASAP" if not mentioned), "priority": "HIGH"|"MEDIUM"|"LOW"}
Maximum 5 items. Order by priority descending.
No markdown fences. No explanation.
""".strip()

meeting_notes = """
Sync notes — 24 June

Priya will fix the login timeout bug before the Friday release — it's blocking QA.
Carlos needs to update the API docs for v2 endpoints, no hard deadline but ideally this sprint.
The whole team should review the new deployment checklist by EOD tomorrow.
Amir to schedule a 30-min session with the data team about the analytics pipeline slowdown — flagged as urgent by the CEO.
Nina volunteered to write the onboarding guide for new contractors, no rush.
""".strip()

resp = router.generate_response(
    meeting_notes,
    model=MODEL,
    system=system,
    max_tokens=400,
    temperature=0,
    stop_sequences=["]"],
)

# The stop sequence cuts before ']', so we restore it
raw = resp.text.strip() + "]"

print("Raw output + restored stop sequence:")
print(raw)
print()

try:
    items = json.loads(raw)
    print(f"Parsed {len(items)} action items:")
    print(f"{'Owner':<10}  {'Priority':<8}  {'Deadline':<10}  Action")
    print("-" * 75)
    for item in items:
        print(f"{item['owner']:<10}  {item['priority']:<8}  {item['deadline']:<10}  {item['action']}")
except json.JSONDecodeError as e:
    print(f"Parse error: {e}")

print()
print(f"Stop reason: {resp.stop_reason}  |  Output tokens: {resp.output_tokens}")
print("Five controls, zero post-processing, directly insertable into a task tracker.")

## Visualising the output control levers

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Map each control lever to: where it lives, what it governs, and when to reach for it
levers = [
    ("Format",         "Prompt",      "Shape of output\n(prose/JSON/table/code)",    0.92),
    ("JSON schema",    "Prompt",      "Parseable, typed\nstructured data",             0.95),
    ("Length spec",    "Prompt",      "Word / sentence /\nbullet count",              0.88),
    ("Tone",           "System",      "Register, formality,\nvoice",                 0.85),
    ("Stop sequence",  "API param",   "Exact generation\nboundary",                  0.80),
    ("Temperature",    "API param",   "Determinism vs\ncreativity",                  0.78),
    ("max_tokens",     "API param",   "Hard ceiling on\noutput length",              0.60),
]

colour_map = {"Prompt": "steelblue", "System": "seagreen", "API param": "tomato"}
labels     = [l[0] for l in levers]
scores     = [l[3] for l in levers]
colours    = [colour_map[l[1]] for l in levers]
descs      = [l[2] for l in levers]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(labels, scores, color=colours, alpha=0.87, height=0.55)

for i, (bar, desc) in enumerate(zip(bars, descs)):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            desc, va='center', fontsize=8, color='#333')

ax.set_xlim(0, 1.45)
ax.set_xlabel("Practical control strength (illustrative)")
ax.set_title("Output control levers — where they live and what they govern")
ax.axvline(0.8, color='grey', linewidth=0.7, linestyle='--', alpha=0.5)
ax.text(0.81, -0.6, "reliable threshold", fontsize=8, color='grey')

patches = [mpatches.Patch(color=c, label=loc) for loc, c in colour_map.items()]
ax.legend(handles=patches, title="Where it lives", fontsize=9, loc='lower right')

plt.tight_layout()
plt.show()

print("Prompt-level controls (format, schema, length) are the most powerful — they shape what the model aims for.")
print("API-level controls (stop sequences, temperature, max_tokens) enforce boundaries after the model starts generating.")
print("Use both layers together: prompt to aim, API params to fence.")

## Key takeaways

- **Format must be specified** — the model defaults to whatever format it last saw in training context. Name the format and describe the schema: "markdown table with columns X, Y, Z" beats "respond in markdown".
- **JSON output** makes model responses directly usable in code. Always declare the schema in the system prompt and always validate with `json.loads()` — never trust it raw.
- **Length specs in the prompt** (word count, sentence count, bullet count) control output length far better than `max_tokens` alone. `max_tokens` is a ceiling, not a target.
- **Tone is specifiable** with three anchors: formality level, stance, and an optional voice reference. The more concrete the description, the more recognisable the register.
- **Stop sequences** give you a precise generation boundary — useful for extracting structured data from mixed-format responses without post-processing.
- **Temperature = 0** for anything parsed programmatically. Higher temperature for brainstorming and creative variation. Never above 1 in production pipelines.
- **Prompt-level controls aim; API-level controls fence.** Use both layers together for reliable, directly usable output.

---

Next up: **Chain-of-Thought Prompting** — when controlling the output shape is not enough and you need the model to reason step by step before it commits to an answer.